In [2]:
import pandas as pd
import numpy as np
from rouge_score import rouge_scorer # For Self-Contrast Metric calculation

# ---------------------------------------------------------
# 1. SETUP: SIMPLE QA DATABASE (Simulating Known & Evolving Data)
# KoLA emphasizes using both "Known" (e.g., Wikipedia) and 
# "Evolving" (e.g., recent news).
# ---------------------------------------------------------

data = {
    "task_id": ["1-1", "2-4", "3-1", "4-1"],
    "level": ["Memorization", "Understanding", "Applying", "Creating"],
    "source_type": ["Known", "Known", "Known", "Evolving"],
    "context": [
        "N/A", 
        "Agrippa defeated Sextus in 36 BC.", 
        "Jeremy Theobald and Christopher Nolan share what profession?", 
        "The Battle of Evesham marked the defeat of Simon de Montfort..."
    ],
    "reference_R": [
        "poet", 
        "Agrippa:person; Sextus:person", 
        "producer", 
        "de Montfort was killed."
    ],
    "annotated_knowledge_K": [
        "N/A", "N/A", "N/A", 
        "Event: Killed; Victim: de Montfort"
    ]
}
df_tasks = pd.DataFrame(data)

# ---------------------------------------------------------
# 2. DEFINING THE FOUR COGNITIVE LEVELS
# Based on the paper's Bloom-inspired taxonomy.
# ---------------------------------------------------------

def test_knowledge_levels(model_outputs):
    """
    KM: Memorization - Recalling facts.
    KU: Understanding - Extracting relations/entities [8].
    KA: Applying - Multi-hop reasoning [8].
    KC: Creating - Generating novel, reasonable knowledge [6].
    """
    pass # In practice, this would iterate through df_tasks

# ---------------------------------------------------------
# 3. SELF-CONTRAST METRIC (Level 4: Creating)
# This distinguishes correctly created knowledge from hallucinations.
# It compares Free Completion (T) vs Grounded Completion (Tk).
# ---------------------------------------------------------

def calculate_self_contrast(T, Tk, R):
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    
    # Delta(A, B) is Rouge-L F1 similarity [11]
    d_TR = scorer.score(R, T)['rougeL'].fmeasure   # Conventional similarity
    d_TTk = scorer.score(Tk, T)['rougeL'].fmeasure # Self-contrast (consistency)
    d_TkR = scorer.score(R, Tk)['rougeL'].fmeasure # Grounded quality
    
    # KoLA Overall Creation Score (x) [11]
    x = (d_TR + d_TTk + d_TkR) / 3
    return x

# ---------------------------------------------------------
# 4. STANDARDIZED SCORING SYSTEM
# KoLA uses z-scores and Min-Max scaling to make metrics 
# comparable across tasks [4, 12].
# ---------------------------------------------------------

def calculate_standardized_scores(raw_scores):
    """
    zij = (xij - mean) / std_dev [4]
    sij = 100 * (zij - min(z)) / (max(z) - min(z)) [4, 9]
    """
    raw_scores = np.array(raw_scores)
    mean = np.mean(raw_scores)
    std = np.std(raw_scores) if np.std(raw_scores) > 0 else 1
    
    z_scores = (raw_scores - mean) / std
    
    # Min-Max Scaling to 0-100 range [4]
    min_z, max_z = np.min(z_scores), np.max(z_scores)
    if max_z == min_z: return [50.0] * len(z_scores)
    
    standard_scores = 100 * (z_scores - min_z) / (max_z - min_z)
    return standard_scores

# ---------------------------------------------------------
# 5. EXECUTION EXAMPLE
# ---------------------------------------------------------

# Mock raw accuracy/F1 scores for 3 different LLMs across 4 tasks
model_results = {
    "Model_A": [0.85, 0.70, 0.45, 0.60], # KM, KU, KA, KC
    "Model_B": [0.60, 0.55, 0.30, 0.40],
    "Model_C": [0.40, 0.40, 0.20, 0.30]
}

# Standardize results per task across models
final_table = []
for i in range(4): # For each task level
    task_raw_scores = [model_results[m][i] for m in model_results]
    standardized = calculate_standardized_scores(task_raw_scores)
    final_table.append(standardized)

df_final = pd.DataFrame(np.array(final_table).T, 
                        index=model_results.keys(), 
                        columns=["KM", "KU", "KA", "KC"])

print("Standardized KoLA Scores (0-100):")
print(df_final)

Standardized KoLA Scores (0-100):
                 KM     KU     KA          KC
Model_A  100.000000  100.0  100.0  100.000000
Model_B   44.444444   50.0   40.0   33.333333
Model_C    0.000000    0.0    0.0    0.000000
